In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('darkgrid')

## Main df

In [2]:
# Import data
df_main = pd.read_excel('../data/df_main.xlsx', index_col=0).reset_index(drop=True)
## Drop unused columns
# drop_columns = ['voltage', 'long_impulse_duration', 'long_impulse_dur_binary']
# df_main = df_main.drop(drop_columns, axis=1)
display(df_main.head())
df_main.info()

,test,one_drop,splashing,breaking_up,net_impact,rebound,voltage,long_impulse_duration,height,inclination,...,roughness,long_impulse_dur_binary,roughness_binary,volume_fraction_binary,velocity,Re,We,We_Re,particle_droplet_diameter_ratio,particle_diameter_cat
0,3,1,1,0,0,0,105.0,10,0.8,0,...,0.10,low,0,1,3.961141,1492.516020,1492.302356,240.108847,0.013301,small
1,4,1,1,0,0,0,105.0,10,0.8,0,...,0.10,low,0,1,3.961141,1492.516020,1492.302356,240.108847,0.013301,small
2,5,1,1,0,0,0,105.0,10,0.8,0,...,0.10,low,0,1,3.961141,1492.516020,1492.302356,240.108847,0.013301,small
3,7,0,1,0,0,0,105.0,10,0.8,0,...,0.04,low,0,1,3.961141,1435.111557,1434.906112,233.148786,0.013833,small
4,8,0,1,0,0,0,105.0,10,0.8,0,...,0.04,low,0,1,3.961141,1435.111557,1434.906112,233.148786,0.013833,small


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 372 entries, 0 to 371
Data columns (total 28 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   test                             372 non-null    int64  
 1   one_drop                         372 non-null    int64  
 2   splashing                        372 non-null    int64  
 3   breaking_up                      372 non-null    int64  
 4   net_impact                       372 non-null    int64  
 5   rebound                          372 non-null    int64  
 6   voltage                          372 non-null    float64
 7   long_impulse_duration            372 non-null    int64  
 8   height                           372 non-null    float64
 9   inclination                      372 non-null    int64  
 10  droplet_diameter                 372 non-null    float64
 11  liquid_density                   372 non-null    int64  
 12  surface_tension       

In [3]:
label_name = ['net_impact', 'splashing', 'breaking_up', 'rebound']

class_count_dict = {}

for label in label_name:
    class_count_dict[label] = df_main[df_main[label]==1].shape[0]

class_count_s = pd.Series(class_count_dict)
class_count_s

net_impact     121
splashing      174
breaking_up    101
rebound         99
dtype: int64

**Labels description:**
- **splashing**: when *'Number of detached small droplets during Spreading'=='many' (more than 5)*;
- **breaking_up**: combines clear "breaking_up" during receding *'Droplet Receding'==2*, as well as the Spreading detaching *0<'Number of detached small droplets during Spreading'<=5*;
- **rebound**: combines all rebound types (partial and total, *'Rebound'>0*), as well as the central jet ejection *'Rim merging or Central jet ejecting'==2*
- **net_impact**: when there is 
    - **no Splashing** (no small droplets detached during spreading, *'Number of detached small droplets during Spreading'==0*), 
    - **no Breaking up** (*'Droplet Receding'<2* [equivalent is *'Number of detached small droplets during Receding or Rim merging' == 0*])
    - **no Rebound** (*'Rebound' == 0*)

**Total count of clear experiments is *372***

## Suspension Data Labeling Edited. For the classes reconfiguration

In [4]:
df_labels = pd.read_excel('../data/Suspension Data Labeling Edited.xlsx', index_col=0)
df_labels = df_labels.drop(index=0)
# Keep only used 372 tests (without draft substrated and liquids)
df_labels = df_labels[df_labels['Test #'].isin(df_main.test.values)]
df_labels = df_labels.drop('Comments', axis=1)
df_labels = df_labels.rename({'Test #': 'test'}, axis=1)
df_labels['test'] = df_labels['test'].astype(int)
df_labels = df_labels.reset_index(drop=True)
display(df_labels)
display(df_labels.info())
df_labels.columns

,test,Total number of dropped drops,Number of detached small droplets during Spreading,Droplet Receding,Rim merging or Central jet ejecting,Number of detached small droplets during Receding or Rim merging,Rebound,Number of detached droplets during Rebound,Final droplets count
0,3,1,many,0,0,0,0,0,1
1,4,1,many,0,0,0,0,0,1
2,5,1,many,0,0,0,0,0,1
3,7,0,many,1,0,0,0,0,2
4,8,0,many,0,0,0,0,0,2
...,...,...,...,...,...,...,...,...,...
367,391,0,0,1,2,0,2,0,0
368,392,1,3,1,1,0,0,0,1
369,393,1,2,1,1,0,0,0,3
370,394,1,0,1,1,1,0,0,2


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 372 entries, 0 to 371
Data columns (total 9 columns):
 #   Column                                                            Non-Null Count  Dtype 
---  ------                                                            --------------  ----- 
 0   test                                                              372 non-null    int32 
 1   Total number of dropped drops                                     372 non-null    object
 2   Number of detached small droplets during Spreading                372 non-null    object
 3   Droplet Receding                                                  372 non-null    object
 4   Rim merging or Central jet ejecting                               372 non-null    object
 5   Number of detached small droplets during Receding or Rim merging  372 non-null    object
 6   Rebound                                                           372 non-null    object
 7   Number of detached droplets during Rebound   

None

Index(['test', 'Total number of dropped drops',
       'Number of detached small droplets during Spreading',
       'Droplet Receding', 'Rim merging or Central jet ejecting',
       'Number of detached small droplets during Receding or Rim merging',
       'Rebound', 'Number of detached droplets during Rebound',
       'Final droplets count'],
      dtype='object')


| Parameter name | Description | Possible values |
| --- | --- | --- |
| Total number of dropped drops | The total number of drops that fell from above | 1, 2 |
| Number of detached small droplets during *spreading* | The number of small droplets that flew out of the lamella drops in the process of spreading | 0, 1, .., 5, many |
| Droplet Receding | Presence (1,2), absence — 0 of a visible drop shrinkage, and an increase in the drop height after reaching the maximum spot diameter. 1 — the usual reduction of the main drop, 2 — the destruction of the main drop in the process of reduction | 0, 1, 2 |
| Rim merging or Central jet ejecting | The presence (1,2), absence — 0 of an explicit rim merging at the end of the shrinkage of the drop (1) and/or the reverse vertical jet (2) (against gravity) | 0, 1, 2 |
| Number of detached small droplets during *Receding or Rim merging* | The number of small droplets separating from the rim of the main drop (rim) and remaining on the substrate during the shrinkage of the drop (receding). Drops do not separate — 0, several drops, many drops — many | 0, 1, .., 5, many |
| Rebound | Presence (1,2), absence — 0 of a drop rebound after the formation of a reverse vertical jet (central jet ejecting). Partial rebound — 1, total rebound - 2 | 0, 1, 2 |
| Number of detached droplets during *Rebound* | The number of droplets separating from the reverse vertical jet (central jet) | 0, 1, .., 5, many |
| Final droplets count | The total number of droplets on the substrate, clearly separated from each other | 1, .., 5, many |

# Show strange tests

Let us consider non-splashing tests when We_Re larger than 150 and splashing before 150.
Do they have correct labels?

On the boundary

In [17]:
df_main[(df_main['splashing'] == 0) & (df_main['We_Re']>150) & (df_main['We_Re']<160)]

,test,one_drop,splashing,breaking_up,net_impact,rebound,voltage,long_impulse_duration,height,inclination,...,roughness,long_impulse_dur_binary,roughness_binary,volume_fraction_binary,velocity,Re,We,We_Re,particle_droplet_diameter_ratio,particle_diameter_cat
38,51,0,0,1,0,0,106.5,12,0.8,0,...,0.10,low,0,1,3.961141,677.852440,913.477171,154.217138,0.012388,small
40,53,0,0,0,1,0,105.0,12,0.8,0,...,0.04,low,0,1,3.961141,677.852440,913.477171,154.217138,0.012388,small
41,54,0,0,0,1,0,105.0,10,0.8,0,...,0.04,low,0,1,3.961141,677.852440,913.477171,154.217138,0.012388,small
43,56,0,0,1,0,1,106.0,12,0.8,0,...,2.49,low,0,1,3.961141,677.852440,913.477171,154.217138,0.012388,small
46,59,1,0,1,0,0,107.0,12,0.8,0,...,2.49,low,0,1,3.961141,693.028241,933.928153,156.799425,0.012117,small
47,60,1,0,1,0,0,107.0,12,0.8,0,...,0.04,low,0,1,3.961141,693.028241,933.928153,156.799425,0.012117,small
49,63,0,0,0,1,0,109.0,12,0.8,0,...,0.10,low,0,1,3.961141,653.571158,880.755601,150.055157,0.012848,small
53,69,1,0,0,0,1,108.0,15,0.8,0,...,2.49,high,0,1,3.961141,682.304008,919.476126,154.976093,0.012307,small
54,70,1,0,1,0,1,103.5,21,0.8,0,...,2.49,high,0,1,3.961141,682.304008,919.476126,154.976093,0.012307,small
60,78,1,0,1,0,0,108.0,15,0.8,0,...,0.10,high,0,1,3.961141,687.969640,927.111159,155.940249,0.040441,medium


### Outliers

In [18]:
df_main[(df_main['splashing'] == 0) & (df_main['We_Re']>160)]

,test,one_drop,splashing,breaking_up,net_impact,rebound,voltage,long_impulse_duration,height,inclination,...,roughness,long_impulse_dur_binary,roughness_binary,volume_fraction_binary,velocity,Re,We,We_Re,particle_droplet_diameter_ratio,particle_diameter_cat
8,16,1,0,1,0,0,103.0,18,0.8,0,...,0.10,high,0,1,3.961141,11274.017403,634.484896,259.555550,0.046453,medium
9,17,0,0,1,0,0,103.0,21,0.8,0,...,0.10,high,0,1,3.961141,11559.676628,650.561372,264.472523,0.045305,medium
10,18,0,0,1,0,0,103.0,18,0.8,0,...,0.04,high,0,1,3.961141,11559.676628,650.561372,264.472523,0.045305,medium
11,19,0,0,0,1,0,103.0,18,0.8,0,...,0.04,high,0,1,3.961141,11559.676628,650.561372,264.472523,0.045305,medium
17,26,1,0,1,0,0,108.0,10,0.8,0,...,0.04,low,0,1,3.961141,11426.368989,643.059016,262.181762,0.045833,medium
18,27,0,0,1,0,0,108.0,10,0.8,0,...,0.10,low,0,1,3.961141,11883.423749,668.781377,270.008528,0.044071,medium
70,91,1,0,0,1,0,112.5,12,0.8,0,...,0.04,low,0,0,3.961141,12662.760711,712.641300,283.183271,0.012483,small
72,93,1,0,0,1,0,114.0,12,0.8,0,...,0.04,low,0,0,3.961141,12645.181682,711.651978,282.888374,0.012500,small
73,94,1,0,0,1,0,114.0,12,0.8,0,...,0.10,low,0,0,3.961141,13445.027511,756.666109,296.205123,0.011756,small
74,95,1,0,0,1,0,114.0,12,0.8,0,...,0.10,low,0,0,3.961141,13787.818581,775.957880,301.851240,0.011464,small


New types in "Number of detached small droplets during Spreading":

- **many small** - when tiny droplets or even particles are detached from the droplet
- **gear** - when branches appear, without explicit detaching.

If "many small" case, gear is not written


TODO: Continue from **351**